In [11]:
import sys
from pathlib import Path


PROJECT_ROOT = Path(r"C:\Users\Sherif\Desktop\ECG analysis")

sys.path.insert(0, str(PROJECT_ROOT))

In [12]:
import numpy as np
import importlib
import src.train_autoencoder

importlib.reload(src.train_autoencoder)

import torch
import matplotlib.pyplot as plt

from models.UNet1D import UNet1D
from src.dataset import create_dataloaders
from src.train_autoencoder import train_autoencoder, compute_lead_reconstruction_matrix

In [13]:
import random
import numpy as np
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [14]:
train_loader, val_loader, test_loader, label_cols, _ = create_dataloaders()

In [15]:
model = UNet1D(in_channels=1, out_channels=12, embedding_dim=32)

In [16]:
train_autoencoder(
    model,
    train_loader,
    val_loader,
    epochs=15,
    lr=1e-3
)


Epoch 1/15


Evaluating AE: 100%|██████████| 69/69 [00:47<00:00,  1.46it/s]


Train Loss (MSE): 0.5647
Val MSE: 0.5510
Val Corr: 0.6307

Epoch 2/15


Evaluating AE: 100%|██████████| 69/69 [00:36<00:00,  1.89it/s]


Train Loss (MSE): 0.5311
Val MSE: 0.5467
Val Corr: 0.6372

Epoch 3/15


Evaluating AE: 100%|██████████| 69/69 [00:36<00:00,  1.88it/s]


... (21 outputs left out)

Evaluating AE: 100%|██████████| 69/69 [00:47<00:00,  1.45it/s]


Train Loss (MSE): 0.4713
Val MSE: 0.4847
Val Corr: 0.6917

Epoch 15/15


Evaluating AE: 100%|██████████| 69/69 [00:52<00:00,  1.31it/s]

Train Loss (MSE): 0.4682
Val MSE: 0.4733
Val Corr: 0.6987


In [17]:
torch.save(model, "unetecg.pth")

In [ ]:
# Reload the saved model (allows running this section independently)
import torch
import numpy as np
from models.UNet1D import UNet1D

model = torch.load('unetecg.pth', map_location='cpu')
model.eval()
print('Model loaded successfully.')

## Test : Matrice de reconstruction lead vers lead (12x12)

Pour chaque lead source i (entree), on calcule le MSE et la correlation de Pearson du lead cible j reconstruit.  
La diagonale (i = j) est **masquee** car reconstruire un lead depuis lui-meme n'est pas informatif.

In [ ]:
# Compute the 12x12 reconstruction matrix (all lead-source -> lead-target pairs)
mse_matrix, corr_matrix = compute_lead_reconstruction_matrix(model, test_loader)
print('Matrix shape:', mse_matrix.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

# Heatmap MSE
fig, ax = plt.subplots(figsize=(11, 9))

masked_mse = np.ma.masked_invalid(mse_matrix)
im = ax.imshow(masked_mse, cmap='RdYlGn_r', aspect='auto',
               vmin=np.nanpercentile(mse_matrix, 5),
               vmax=np.nanpercentile(mse_matrix, 95))

ax.set_xticks(range(12))
ax.set_yticks(range(12))
ax.set_xticklabels(LEAD_NAMES, fontsize=10)
ax.set_yticklabels(LEAD_NAMES, fontsize=10)
ax.set_xlabel('Lead reconstruit (cible j)', fontsize=12)
ax.set_ylabel('Lead entree (source i)', fontsize=12)
ax.set_title('Matrice de reconstruction -- MSE\n'
             '(diagonale masquee : reconstruction triviale du meme lead)', fontsize=13)

for i in range(12):
    for j in range(12):
        if i != j:
            ax.text(j, i, f'{mse_matrix[i, j]:.3f}',
                    ha='center', va='center', fontsize=7,
                    color='black' if masked_mse[i, j] < np.nanmean(mse_matrix) else 'white')
        else:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                         fill=True, color='lightgray', zorder=2))
            ax.text(j, i, 'N/A', ha='center', va='center', fontsize=7, color='gray', zorder=3)

plt.colorbar(im, ax=ax, label='MSE')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap Correlation de Pearson
fig, ax = plt.subplots(figsize=(11, 9))

masked_corr = np.ma.masked_invalid(corr_matrix)
im = ax.imshow(masked_corr, cmap='RdYlGn', aspect='auto',
               vmin=np.nanpercentile(corr_matrix, 5),
               vmax=np.nanpercentile(corr_matrix, 95))

ax.set_xticks(range(12))
ax.set_yticks(range(12))
ax.set_xticklabels(LEAD_NAMES, fontsize=10)
ax.set_yticklabels(LEAD_NAMES, fontsize=10)
ax.set_xlabel('Lead reconstruit (cible j)', fontsize=12)
ax.set_ylabel('Lead entree (source i)', fontsize=12)
ax.set_title('Matrice de reconstruction -- Correlation de Pearson\n'
             '(diagonale masquee : reconstruction triviale du meme lead)', fontsize=13)

for i in range(12):
    for j in range(12):
        if i != j:
            ax.text(j, i, f'{corr_matrix[i, j]:.3f}',
                    ha='center', va='center', fontsize=7,
                    color='black' if masked_corr[i, j] > np.nanmean(corr_matrix) else 'white')
        else:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                         fill=True, color='lightgray', zorder=2))
            ax.text(j, i, 'N/A', ha='center', va='center', fontsize=7, color='gray', zorder=3)

plt.colorbar(im, ax=ax, label='Correlation de Pearson')
plt.tight_layout()
plt.show()

In [ ]:
# Analyse par lead : quel lead source donne la meilleure reconstruction ?
mean_mse_per_source = np.nanmean(mse_matrix, axis=1)   # moyenne par ligne (source)
mean_mse_per_target = np.nanmean(mse_matrix, axis=0)   # moyenne par colonne (cible)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(LEAD_NAMES, mean_mse_per_source, color='steelblue')
axes[0].set_title('MSE moyen par lead SOURCE (entree)', fontsize=12)
axes[0].set_xlabel('Lead source')
axes[0].set_ylabel('MSE moyen')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(LEAD_NAMES, mean_mse_per_target, color='coral')
axes[1].set_title('MSE moyen par lead CIBLE (sortie)', fontsize=12)
axes[1].set_xlabel('Lead cible')
axes[1].set_ylabel('MSE moyen')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Analyse des embeddings (version corrigee)

**Approche :** Pour chaque ECG du test set, on passe les 12 leads originaux **un par un** comme entree.  
On obtient ainsi 12 versions de l'embedding (une par lead source).  
On **moyenne ces 12 versions** pour obtenir un embedding representatif de l'ECG, puis on applique PCA -> UMAP.

> Cette approche est coherente avec la litterature : chaque embedding moyen capture la meme representation vue depuis 12 perspectives differentes.

In [ ]:
# Collecte des embeddings (12 versions par ECG, moyennees)
N_SAMPLES = 500  # sous-ensemble pour la vitesse

model.eval()
all_embeddings = []
all_labels     = []
n_collected    = 0

with torch.no_grad():
    for signals, labels in test_loader:
        if n_collected >= N_SAMPLES:
            break

        signals = signals.to('cpu')

        # Pour chaque ECG du batch, passer chacun des 12 leads en entree
        emb_versions = []
        for i in range(12):
            input_lead = signals[:, i:i+1, :]
            _, embeddings = model(input_lead)  # (B, 12, D)
            emb_versions.append(embeddings.cpu().numpy())

        # Stack -> (12_versions, B, 12_leads, D)
        emb_versions = np.stack(emb_versions, axis=0)

        # Moyenne sur les 12 versions -> (B, 12_leads, D)
        mean_emb = emb_versions.mean(axis=0)

        all_embeddings.append(mean_emb)
        all_labels.append(labels.numpy())
        n_collected += signals.shape[0]

all_embeddings = np.concatenate(all_embeddings, axis=0)[:N_SAMPLES]  # (N, 12, D)
all_labels     = np.concatenate(all_labels,     axis=0)[:N_SAMPLES]  # (N, 5)

# Aplatir : (N, 12*D)
X = all_embeddings.reshape(N_SAMPLES, -1)
print(f'Embeddings shape : {X.shape}')
print(f'Labels shape     : {all_labels.shape}')

In [ ]:
# PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)
print(f'Variance expliquee (50 composantes) : {pca.explained_variance_ratio_.sum():.3f}')

In [ ]:
# UMAP
import umap

reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_pca)
print('UMAP termine. Shape :', X_umap.shape)

In [ ]:
# Visualisation UMAP par superclasse
SUPERCLASSES = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
COLORS = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for i, (sc, color) in enumerate(zip(SUPERCLASSES, COLORS)):
    mask = all_labels[:, i] == 1
    axes[i].scatter(X_umap[~mask, 0], X_umap[~mask, 1],
                    c='lightgray', alpha=0.4, s=10, label='Autres')
    axes[i].scatter(X_umap[mask, 0], X_umap[mask, 1],
                    c=color, alpha=0.8, s=15, label=sc)
    axes[i].set_title(f'UMAP -- {sc}', fontsize=11)
    axes[i].legend(markerscale=2, fontsize=8)
    axes[i].set_xlabel('UMAP 1')
    axes[i].set_ylabel('UMAP 2')

plt.suptitle(
    'Embeddings moyennes (12 leads) -- projection UMAP par superclasse',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()